# Lab 04: Quantum Circuits

**Goal:** Master circuit construction with various gates and understand the simulation workflow.

---

In [ ]:
# Install packages and force Python to recognize them
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "qiskit", "qiskit-aer", "pylatexenc", "matplotlib", "-q"])

import importlib
importlib.invalidate_caches()
import pylatexenc

from qiskit.utils import optionals
if hasattr(optionals, 'HAS_PYLATEXENC'):
    optionals.HAS_PYLATEXENC._is_available = True

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np
print("Ready!")

## Exercise 1: Building Circuits Step by Step

In [ ]:
# Create an empty circuit with 3 qubits and 3 classical bits
qc = QuantumCircuit(3, 3)

# Add gates one by one
qc.h(0)         # Hadamard on qubit 0
qc.x(1)         # X (NOT) on qubit 1
qc.cx(0, 2)     # CNOT: control=0, target=2
qc.barrier()    # Visual separator (no effect on computation)
qc.measure([0, 1, 2], [0, 1, 2])

qc.draw('mpl')

In [ ]:
simulator = AerSimulator()
result = simulator.run(qc, shots=1000).result()
plot_histogram(result.get_counts())

## Exercise 2: Common Gate Reference

In [ ]:
# Demonstrate each common gate
def demo_gate(gate_name, circuit_func):
    qc = QuantumCircuit(2, 2)
    circuit_func(qc)
    qc.measure([0, 1], [0, 1])
    result = simulator.run(qc, shots=1000).result()
    print(f"{gate_name}: {result.get_counts()}")
    return qc

# X gate (NOT)
print("X Gate - Flips |0⟩ to |1⟩")
qc = demo_gate("X(0)", lambda qc: qc.x(0))

# H gate (Hadamard)
print("\nH Gate - Creates superposition")
qc = demo_gate("H(0)", lambda qc: qc.h(0))

# Z gate (Phase flip)
print("\nZ Gate - Phase flip (no visible effect on |0⟩ state)")
qc = demo_gate("Z(0)", lambda qc: qc.z(0))

In [ ]:
# CNOT gate demonstration
print("CNOT Gate - Flips target if control is |1⟩")
print("\nControl=|0⟩:")
qc = QuantumCircuit(2, 2)
qc.cx(0, 1)  # Control=0, Target=1
qc.measure([0, 1], [0, 1])
result = simulator.run(qc, shots=100).result()
print(f"  Result: {result.get_counts()}")

print("\nControl=|1⟩:")
qc = QuantumCircuit(2, 2)
qc.x(0)      # Set control to |1⟩
qc.cx(0, 1)  # Now target will flip
qc.measure([0, 1], [0, 1])
result = simulator.run(qc, shots=100).result()
print(f"  Result: {result.get_counts()}")

## Exercise 3: Rotation Gates

In [ ]:
# RY gate - rotate around Y axis
angles = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi]

print("RY Gate at different angles:")
for angle in angles:
    qc = QuantumCircuit(1, 1)
    qc.ry(angle, 0)
    qc.measure(0, 0)
    result = simulator.run(qc, shots=1000).result()
    counts = result.get_counts()
    p1 = counts.get('1', 0) / 1000
    print(f"  theta={angle:.2f}: P(1)={p1:.2f}")

## Exercise 4: Circuit Composition

In [ ]:
# Build a reusable sub-circuit
def create_bell_pair():
    """Creates a Bell state on 2 qubits"""
    qc = QuantumCircuit(2, name='Bell')
    qc.h(0)
    qc.cx(0, 1)
    return qc.to_gate()

# Use it in a larger circuit
main_circuit = QuantumCircuit(4, 4)
bell = create_bell_pair()

# Create two Bell pairs
main_circuit.append(bell, [0, 1])
main_circuit.append(bell, [2, 3])
main_circuit.measure_all()

main_circuit.draw('mpl')

In [ ]:
# Decompose the custom gate so Aer can run it
decomposed_circuit = main_circuit.decompose()
result = simulator.run(decomposed_circuit, shots=1000).result()
plot_histogram(result.get_counts())

## Exercise 5: Controlled Gates

In [ ]:
# Toffoli gate (CCX) - CNOT with 2 controls
qc = QuantumCircuit(3, 3)
qc.x(0)  # Set control 1 to |1⟩
qc.x(1)  # Set control 2 to |1⟩
qc.ccx(0, 1, 2)  # Target flips only if BOTH controls are |1⟩
qc.measure([0, 1, 2], [0, 1, 2])

qc.draw('mpl')

In [ ]:
result = simulator.run(qc, shots=100).result()
print("Toffoli result (both controls=1):", result.get_counts())

## Challenge: Build a Quantum Adder

Create a circuit that adds two 1-bit numbers.

In [ ]:
# Half adder: a + b = (sum, carry)
# sum = a XOR b (CNOT)
# carry = a AND b (Toffoli, but we need ancilla)

def half_adder(a, b):
    """Add two bits, return (sum, carry)"""
    qc = QuantumCircuit(3, 2)  # a, b, carry_out
    
    # Set inputs
    if a: qc.x(0)
    if b: qc.x(1)
    
    # Compute carry = a AND b
    qc.ccx(0, 1, 2)
    
    # Compute sum = a XOR b
    qc.cx(0, 1)  # Result in qubit 1
    
    # Measure sum (qubit 1) and carry (qubit 2)
    qc.measure(1, 0)  # sum
    qc.measure(2, 1)  # carry
    
    return qc

# Test all combinations
print("Half Adder Results:")
print("-" * 30)
for a in [0, 1]:
    for b in [0, 1]:
        qc = half_adder(a, b)
        result = simulator.run(qc, shots=100).result()
        outcome = list(result.get_counts().keys())[0]
        carry, sum_bit = int(outcome[0]), int(outcome[1])
        print(f"{a} + {b} = {carry}{sum_bit} (decimal: {a+b})")